In [3]:
from tdc.single_pred import ADME
data = ADME(name='Solubility_AqSolDB')
split = data.get_split(method = 'scaffold')

train = split['train']
print(train.columns.tolist())
print(train.head())
print({k: len(v) for k, v in split.items()})
print(train['Y'].describe())

#Y is the log solubility, Continuous - Most durg-like molecules dissolve at concentrataions below 1 hence log < 1
#The sign is just saying "less than one mole per litre" solubility expressed in mol/L
#More negative means less soluble.


Found local copy...
Loading...
Done!
100%|██████████| 9982/9982 [00:01<00:00, 8691.48it/s]

['Drug_ID', 'Drug', 'Y']
                                             Drug_ID  \
0                               4-chlorobenzaldehyde   
1                                       vinyltoluene   
2                      4-(dimethylamino)benzaldehyde   
3               2-methyl-1-phenylpropan-2-yl acetate   
4  5-methoxy-1-[4-(trifluoromethyl)phenyl]pentan-...   

                            Drug         Y  
0                O=Cc1ccc(Cl)cc1 -2.177078  
1                 C=Cc1cccc(C)c1 -3.123150  
2             CN(C)c1ccc(C=O)cc1 -2.282769  
3        CC(=O)OC(C)(C)Cc1ccccc1 -2.394650  
4  COCCCCC(=O)c1ccc(C(F)(F)F)cc1 -3.544060  
{'train': 6987, 'valid': 998, 'test': 1997}
count    6987.000000
mean       -2.578511
std         2.251413
min       -11.998938
25%        -3.950000
50%        -2.349000
75%        -0.962413
max         1.967513
Name: Y, dtype: float64


In [4]:
#split.items()- split is the dictionary, calling .items() on a dict gives its key-value pairs one at a time
#k here is the key, dict. made of key-value pairs so the comprehension has to produce two things on every pass

print({k: len(v) for k, v in split.items()})

{'train': 6987, 'valid': 998, 'test': 1997}


In [5]:
#df gets the value (the DataFrame). On each pass
for name, df in split.items():
    print(name, round(df['Y'].mean(), 2), round(df['Y'].std(), 2))


#Means have spread apart, Under a random split they would be much closer/near identical 
#Valid and test are noticeable more negative meaning the held-out scaffolds are, on average less soluble than the training ones
#std also spreads out menaing valid.  set covers a somewhat broarder range of solublilities

train -2.58 2.25
valid -4.04 2.75
test -3.4 2.3


In [6]:
print(train['Y'].describe())
# train is a pandas Dataframe (a table). Indexing it with ['Y'] pulls out a single column by name. 
# .describe() - pandas method that computes a batch of summary stats for that given pandas series.

count    6987.000000
mean       -2.578511
std         2.251413
min       -11.998938
25%        -3.950000
50%        -2.349000
75%        -0.962413
max         1.967513
Name: Y, dtype: float64


In [7]:
from rdkit import Chem # Cheminformatics library, Chem is the core module

#MolFromSmiles parses the SMILES and builds a proper Mol object - RDKit's internal representation of the molecules 
#Structure, with actual atom objects and bond objects connecting them. This Parse step is the bridge from "text" to "Chemistry that can be queried"

mol = Chem.MolFromSmiles(train['Drug'].iloc[0]) #Grabs the Drug column (smiles string) as a series 
#.iloc[0] - iloc means "index by integer position" and [0] takes the first one, hence pulls SMILES string of the first molecule in the training set

print("atoms",mol.GetNumAtoms()) #now that mol is a real structure, can ask quesions - like "how many atoms"

atoms 9


In [8]:
#mol is a structured object, RDKit parsed the SMILES text and build a Mol object in memory. It's a container that holds, among other things a collection of atom objects and bond objects
# and information about how they are connected.
#mol.GetAtoms() is a method on that object - a function attatched to mol that, when called hands back the collection of atom objects living inside it. 
#It returns somthing you can iterate over, where ech item is a full Atom object, not a letter, a object that itself has methods (e.g GetSymbol(), GetDegree(),etc)
#Pattern: mol.GetSomthing() - asks the whole molecule a question (e.g GetNumAtoms()), atom.GetSomthing() asks a single atom a question (GetSymbol(),GetDegree())


for atom in mol.GetAtoms(): 
    print(atom.GetSymbol(),atom.GetDegree())

O 1
C 2
C 3
C 2
C 2
C 3
Cl 1
C 2
C 2


In [ ]:
def atom_features(atom): #atom in the parenthisis as atom is the parameter - this function needs one input to d its job and i'll call that input "atom"
   #atom_features isn't about one specific atom - its a general procedure for turning any atom into 14 numbers. It needs a slot to receive whichever atom you want processed. That slot is the parameter. 
    allowed = ['C','N','O','F','P','S','Cl','Br','I']
# One-hot slice (length 10: 9 elements + 1 "other")
    symbol = atom.GetSymbol()
    onehot = [0] * (len(allowed) + 1) # Ten zeros 
    if symbol in allowed:
        onehot[allowed.index(symbol)] = 1
    else:
        onehot[-1] = 1 #The "other" bucket. [-1] is the last slot. Any element not on the list lands here producing an al zero-slice. 

    degree = atom.GetDegree()
    charge = atom.GetFormalCharge()
    aromatic = int(atom.GetIsAromatic()) #GetIsAromatic returns a Python True/False, not a number. int(True) = 1
    num_hs = atom.GetTotalNumHs() # adding () calls the function, actually running it 

    return onehot + [degree,charge,aromatic,num_hs]


In [16]:
first_atom = mol.GetAtoms()[0] 
print(atom_features(first_atom))
print(len(atom_features(first_atom)))

[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
14


In [17]:
atom_rows = [atom_features(atom) for atom in mol.GetAtoms()]
#Loop over every atom in the molecule, run the featuriser on each, collect the results into a list. 
#So atoms_rows becomes a list of 9 lists, each of length 14, exactly the matrix shape wanted.
#For atom in mol.GetAtoms() - loops over the molecule's atoms; each loop, the current atom object gets the name atom
#atom_features(atom) - this takes the current atom and passes it into the function
#on each pass, a real atom object flows into the function's placeholder, comes back as 14 numbers an dgets collected into the list. Nine atoms oin -> nine out

#in def atom_features(atom) - atom is a placeholder naminng the input the function expects 
#in atom_features(atom) (when calling) atom is the actual atom you're feeding in

In [19]:
print(len(atom_rows))
print(len(atom_rows[0]))

9
14


In [ ]:
#right now atom_rows is a python list of lists. PyTorch doesn't train on plain lists - it needs a tensor (its own numeric array type)

import torch 
x = torch.tensor(atom_rows, dtype=torch.float)
print(x.shape)

#dtype=torch.float - the network does floating-point math, so we declare these floats (not integers)
#x.shape - tensor knows its own dimensions. seeing that shape is prrof the whole atom-featurization pipeline works end to end 


torch.Size([9, 14])


In [21]:
for bond in mol.GetBonds():
    i = bond.GetBeginAtomIdx()
    j = bond.GetEndAtomIdx()
    print(i,j)

    #mol.GetBonds() - molecule's bonds
    #GetBeginAtomIdx() and GetEndAtomIdx() - the integer indicies of the two atoms a bond connects
    #Indicies, not atom objects, which is what edge_index wants, since it referes to atoms by their row bumber in x

    



0 1
1 2
2 3
3 4
4 5
5 6
5 7
7 8
8 2
